# RankResume Stage 2: Adaptive Groq Questions

This notebook requires explicit `EXAM_ID`, `JD_ID`, and `CANDIDATE_ID`. It reuses a valid four-question batch and never selects a random exam or candidate. Apply `selection_generation_schema.sql` before running it.

In [9]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'Question_Selection_Generation':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

EXAM_ID = 'f37e2dcf-7031-4af8-993b-eeb76c425a77'
JD_ID = 2
CANDIDATE_ID = 2
from Question_Selection_Generation.db_helper import connect
from Question_Selection_Generation.stage2_workflow import generate_or_reuse_stage2
print(JD_ID, CANDIDATE_ID, EXAM_ID)

2 2 f37e2dcf-7031-4af8-993b-eeb76c425a77


In [10]:
if JD_ID <= 0 or CANDIDATE_ID <= 0 or EXAM_ID.startswith('replace-'):
    raise ValueError('Set EXAM_ID, JD_ID, and CANDIDATE_ID explicitly before running this notebook.')
with connect() as connection:
    with connection.cursor() as cursor:
        cursor.execute('SELECT 1 FROM exam_sessions WHERE exam_id = %s AND jd_id = %s', (EXAM_ID, JD_ID))
        if cursor.fetchone() is None:
            raise ValueError('The supplied exam does not belong to the supplied JD.')
        cursor.execute('SELECT COUNT(*) FROM bankresume')
        print('Question Bank rows:', cursor.fetchone()[0])

Question Bank rows: 5194


In [14]:
import importlib
import Question_Selection_Generation.groq_client as groq_client

# Reload the module to pick up the latest code
importlib.reload(groq_client)

print("Configured model:", groq_client.GROQ_MODEL)

try:
    data = groq_client._request("/models")
    print("SUCCESS: Notebook can access Groq.")
    print("Available models:", len(data.get("data", [])))
except Exception as exc:
    print("FAILED:", type(exc).__name__, str(exc))

Configured model: llama-3.3-70b-versatile
SUCCESS: Notebook can access Groq.
Available models: 11


The workflow rejects pending, failed, review-required, incomplete, invalid, duplicate, or unrelated LLM output. Groq availability and rate limits depend on the configured account and model; API keys are loaded from `.env` and are never printed.